# [[Semantic Collect Data Format v2.0]]

In [ ]:
text = "Jammingbot is a fantasy on the theme of a post-apocalyptic future, when the main functions of the Internet and assistant bots will be defeated and only one self-replicating bot will remain, aimlessly plowing the Internet. This is a bot that has no goal, but only a path."

In [ ]:
#
# Semantic Collect Data Format v2.0
#
import spacy
from collections import Counter
from typing import Dict, List, Any

nlp = spacy.load("en_core_web_sm")   # или "ru_core_news_sm" для русского

def analyze_semantic_v2_0(text: str) -> Dict[str, Any]:
    doc = nlp(text)
    
    result = {
        "original_text": text,
        "num_sentences": len(list(doc.sents)),
        "sentences": [],
        "global": {},
        "mood": "neutral",
        "self_referential_score": 0.0,
        "themes": []
    }
    
    all_noun_phrases = []
    all_verbs = []
    all_subjects = []
    
    for sent_id, sent in enumerate(doc.sents):
        sent_doc = nlp(sent.text)  # перепарсим предложение для точности
        
        # Noun phrases
        noun_phrases = [chunk.text.strip() for chunk in sent_doc.noun_chunks 
                       if len(chunk.text.strip()) > 2]
        all_noun_phrases.extend(noun_phrases)
        
        # Verbs (lemmas)
        verbs = [token.lemma_ for token in sent_doc if token.pos_ in ("VERB", "AUX")]
        all_verbs.extend(verbs)
        
        # Entities
        entities = [{"text": ent.text, "label": ent.label_} for ent in sent_doc.ents]
        
        # Simple SVO extraction
        svo = []
        for token in sent_doc:
            if token.dep_ in ("nsubj", "nsubjpass"):
                subj = token.text
                verb = token.head.lemma_
                obj = None
                for child in token.head.children:
                    if child.dep_ in ("dobj", "attr", "pobj", "oprd"):
                        obj = child.text
                svo.append({"subject": subj, "verb": verb, "object": obj})
        
        # Dependencies
        dependencies = [
            {
                "token": token.text,
                "lemma": token.lemma_,
                "pos": token.pos_,
                "dep": token.dep_,
                "head": token.head.text
            }
            for token in sent_doc
        ]
        
        result["sentences"].append({
            "sentence_id": sent_id,
            "text": sent.text.strip(),
            "noun_phrases": noun_phrases,
            "verbs": verbs,
            "entities": entities,
            "svo_triples": svo,
            "dependencies": dependencies
        })
        
        all_subjects.extend([t["subject"] for t in svo])
    
    # Global stats
    result["global"] = {
        "all_noun_phrases": list(dict.fromkeys(all_noun_phrases)),  # unique preserve order
        "all_verbs": list(dict.fromkeys(all_verbs)),
        "main_subjects": [x[0] for x in Counter(all_subjects).most_common(3)],
        "key_phrases": [x[0] for x in Counter(all_noun_phrases).most_common(6)]
    }
    
    # Simple mood and self-referential detection
    mood_keywords = {
        "melancholic-philosophical": ["no goal", "only a path", "aimlessly", "remains", "fantasy"],
        "post-apocalyptic": ["post-apocalyptic", "defeated", "ruins"]
    }
    
    text_lower = text.lower()
    if any(kw in text_lower for kw in mood_keywords["melancholic-philosophical"]):
        result["mood"] = "melancholic-philosophical"
    elif any(kw in text_lower for kw in mood_keywords["post-apocalyptic"]):
        result["mood"] = "post-apocalyptic"
    
    # Self-referential score (очень грубо)
    bot_words = ["bot", "jammingbot", "he", "this is a bot"]
    result["self_referential_score"] = sum(1 for word in bot_words if word.lower() in text_lower) / 4.0
    
    return result

In [ ]:
step_semantic_analyze_v2_0 = analyze_semantic_v2_0(text)
print(step_semantic_analyze_v2_0)

# Draw graph

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx


def build_graph_from_svo_list(svo_list):
    """Small SVO-only graph for matplotlib demo (not the full Source/SVO analysis graph)."""
    G = nx.DiGraph()

    for triple in svo_list:
        subj = triple["subject"].strip()
        verb = triple["verb"].strip()
        obj = triple.get("object")

        if not subj:
            continue

        G.add_node(subj, type="entity")

        if obj:
            obj = obj.strip()
            G.add_node(obj, type="entity")
            G.add_edge(
                subj,
                obj,
                label=verb,
                verb=verb,
                relation_type="action",
            )
        else:
            action_node = f"{subj}_{verb}"
            G.add_node(action_node, type="action")
            G.add_edge(
                subj,
                action_node,
                label=verb,
                verb=verb,
                relation_type="self_action",
            )

    return G


def svo_from_analysis(analysis):
    out = []
    for s in analysis.get("sentences", []):
        out.extend(s.get("svo_triples", []))
    return out


svo_demo = [
    {"subject": "Jammingbot", "verb": "be", "object": "a fantasy"},
    {"subject": "functions", "verb": "defeat", "object": None},
    {"subject": "bot", "verb": "remain", "object": None},
    {"subject": "bot", "verb": "plow", "object": "Internet"},
    {"subject": "This", "verb": "be", "object": "a bot"},
    {"subject": "bot", "verb": "have", "object": "goal"},
]

svo_list = svo_from_analysis(step_semantic_analyze_v2_0)
if not svo_list:
    svo_list = svo_demo

G = build_graph_from_svo_list(svo_list)


def draw_knowledge_graph(G, title="SVO knowledge graph", figsize=(10, 7), seed=42):
    if G.number_of_nodes() == 0:
        print("Graph is empty — nothing to draw.")
        return

    fig, ax = plt.subplots(figsize=figsize)
    pos = nx.spring_layout(G, seed=seed, k=2 / max(G.number_of_nodes(), 1) ** 0.5)

    node_types = nx.get_node_attributes(G, "type")
    entity_nodes = [n for n, t in node_types.items() if t == "entity"]
    action_nodes = [n for n, t in node_types.items() if t == "action"]

    nx.draw_networkx_nodes(
        G, pos, nodelist=entity_nodes, node_color="#6fa8dc", node_size=1400, ax=ax
    )
    nx.draw_networkx_nodes(
        G, pos, nodelist=action_nodes, node_color="#b6d7a8", node_size=1000, ax=ax
    )
    nx.draw_networkx_edges(
        G,
        pos,
        edge_color="#444444",
        arrows=True,
        arrowsize=18,
        ax=ax,
        connectionstyle="arc3,rad=0.08",
        min_source_margin=18,
        min_target_margin=18,
    )
    nx.draw_networkx_labels(G, pos, font_size=9, ax=ax)
    edge_labels = nx.get_edge_attributes(G, "label")
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8, ax=ax)
    ax.set_axis_off()
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


draw_knowledge_graph(G, title=f"SVO graph — mood: {step_semantic_analyze_v2_0['mood']}")

print("Nodes:", list(G.nodes()))
print("Edges (with verb):", [(u, v, d["label"]) for u, v, d in G.edges(data=True)])
print("Degree centrality:", dict(G.degree()))

In [ ]:
import os

from IPython.display import FileLink, display
from pyvis.network import Network


def create_interactive_graph(
    G, filename="jammingbot_knowledge_graph.html", *, show_link=True
):
    # notebook=False here means prep_notebook() is skipped, so self.template stays None.
    # pyvis Network.show() defaults to notebook=True and then uses that None template — use write_html(..., notebook=False) instead.
    net = Network(
        height="800px",
        width="100%",
        directed=True,
        notebook=False,
        cdn_resources="in_line",
    )

    for node in G.nodes():
        node_type = G.nodes[node].get("type", "entity")
        color = "#A8DADC" if node_type == "entity" else "#E9C46A"
        label = str(node)
        size = 25 if "bot" in label.lower() or "jammingbot" in label.lower() else 15
        net.add_node(node, label=label, color=color, size=size)

    for u, v, data in G.edges(data=True):
        label = data.get("label", "")
        net.add_edge(u, v, label=label, title=label)

    net.set_options(
        """
    var options = {
      "physics": {"stabilization": true},
      "nodes": {"font": {"size": 14}}
    }
    """
    )
    out_dir = os.path.dirname(os.path.abspath(filename))
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    net.write_html(filename, open_browser=False, notebook=False)
    path = os.path.abspath(filename)
    print(f"Граф сохранён в {path}")
    if show_link:
        display(FileLink(filename))


create_interactive_graph(G)

In [ ]:
import networkx as nx


def build_knowledge_graph(analysis):
    """
    Строит граф знаний из полного анализа spaCy.

    Принимает:
    - str: сырой текст → внутри вызывается analyze_semantic_v2_0
    - dict: результат analyze_semantic_v2_0, либо API-объект с полем text / original_text
      (если нет sentences — текст снова прогоняется через analyze_semantic_v2_0).
    """
    if isinstance(analysis, str):
        analysis = analyze_semantic_v2_0(analysis)
    elif isinstance(analysis, dict):
        raw = (analysis.get("original_text") or analysis.get("text") or "").strip()
        sentences = analysis.get("sentences") or []
        if not sentences and raw:
            analysis = analyze_semantic_v2_0(raw)
        elif not analysis.get("original_text") and raw:
            analysis = {**analysis, "original_text": raw}
        elif not analysis.get("original_text"):
            analysis = {**analysis, "original_text": ""}
    else:
        raise TypeError(
            "build_knowledge_graph: ожидается str или dict "
            "(JSON анализатора или API с ключом text / original_text)."
        )

    source_text = analysis.get("original_text") or ""
    G = nx.DiGraph()

    # 1. Мета-узел Source
    slice_id = f"slice_{hash(source_text[:150]) % 10000000}"
    preview = (
        source_text[:280] + "..."
        if len(source_text) > 280
        else source_text
    )
    G.add_node(
        slice_id,
        type="Source",
        label="Source",
        original_text=preview,
        mood=analysis.get("mood", "neutral"),
        self_referential_score=analysis.get("self_referential_score", 0.0),
        num_sentences=analysis.get("num_sentences", 0),
        discovery_time=None,
    )
    
    # 2. Обрабатываем SVO из всех предложений
    for sent in analysis.get('sentences', []):
        for svo in sent.get('svo_triples', []):
            subj = str(svo.get('subject', '')).strip()
            verb = str(svo.get('verb', '')).strip().lower()
            obj  = str(svo.get('object', '')).strip() if svo.get('object') is not None else None
            
            if not subj:
                continue
            
            # Нормализация главных сущностей
            if subj.lower() in ['jammingbot', 'bot', 'this', 'that', 'он']:
                subj_node = "Bot"
            else:
                subj_node = subj[:60]   # ограничиваем длину
            
            G.add_node(subj_node, type="Entity")
            
            # Связь от Source к сущности
            G.add_edge(slice_id, subj_node, relation="CONTAINS", verb="contains")
            
            if obj:
                # Нормализация объекта
                if obj.lower() in ['bot', 'this', 'that', 'он']:
                    obj_node = "Bot"
                else:
                    obj_node = obj[:60]
                
                G.add_node(obj_node, type="Entity")
                G.add_edge(subj_node, obj_node,
                          label=verb,
                          verb=verb,
                          relation_type="action",
                          from_sentence=sent['sentence_id'])
            else:
                # Действие без явного объекта
                action_node = f"{subj_node}_{verb}"
                G.add_node(action_node, type="Action", label=verb.upper())
                G.add_edge(subj_node, action_node,
                          label=verb,
                          verb=verb,
                          relation_type="self_action",
                          from_sentence=sent['sentence_id'])
    
    # 3. Добавляем ключевые фразы
    for phrase in analysis.get("global", {}).get("key_phrases", [])[:10]:
        if phrase and len(phrase.strip()) > 3:
            clean_phrase = phrase.strip()[:70]
            G.add_node(clean_phrase, type="KeyPhrase")
            G.add_edge(slice_id, clean_phrase, relation="HAS_KEYPHRASE")
    
    # 4. Добавляем темы (если есть)
    for theme in analysis.get('themes', []):
        if theme:
            G.add_node(theme, type="Theme")
            G.add_edge(slice_id, theme, relation="HAS_THEME")
    
    return G

In [ ]:
# analysis — это твой полный JSON-вывод парсера
G = build_knowledge_graph(step_semantic_analyze_v2_0)

# Посмотреть основные узлы и рёбра
print("Узлы:", list(G.nodes()))
print("Рёбра:")
for u, v, data in G.edges(data=True):
    label = data.get('label', data.get('relation', ''))
    print(f"{u} --[{label}]--> {v}")

In [ ]:
create_interactive_graph(G)

In [ ]:
import json
import os

import requests

STEPS_DIR = "steps"
os.makedirs(STEPS_DIR, exist_ok=True)

for i in range(8, 50):
    url = f"https://storage.jamming-bot.arthew0.online/get/step/{i}"
    print(url)
    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        data = response.json()
    except (requests.RequestException, ValueError) as e:
        print(f"  skip step {i}: {e}")
        continue

    if not data:
        print(f"  skip step {i}: empty JSON")
        continue

    text = str(data.get("text") or "").strip()
    if not text:
        print(f"  skip step {i}: no text (keys: {list(data)[:12]}...)")
        continue

    print(f"Building graph for step {i}: {text[:100]!r}...")
    semantic = analyze_semantic_v2_0(text)
    json_path = os.path.join(STEPS_DIR, f"step_{i}.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(semantic, f, ensure_ascii=False, indent=2)
    print(f"  semantic JSON → {os.path.abspath(json_path)}")

    G = build_knowledge_graph(semantic)
    out_path = os.path.join(STEPS_DIR, f"jammingbot_knowledge_graph_{i}.html")
    create_interactive_graph(G, filename=out_path, show_link=False)

print("Done. HTML + step_*.json under", os.path.abspath(STEPS_DIR))